# Generate a batch of prompts at once

A small frozen decoder written in NumPy reads a batch of token ids, an attention mask and
a position number for every column, and returns the logits of the next token for each
row. Generating for four prompts one at a time takes four times as many calls, so you
will generate for all four in one batch. A batch is right when every row generates
exactly the tokens its prompt generates alone.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first. Token 0 is the pad, and the prompts use only the tokens 1 to 11.

In [ ]:
import numpy as np

V, D, MAX_POS, PAD = 12, 8, 32, 0
rng = np.random.default_rng(17)
E = rng.normal(0, 1, (V, D))           # token embeddings, one row per token id
P = rng.normal(0, 1, (MAX_POS, D))     # learned position embeddings, one row per position number
Wq, Wk, Wv = (rng.normal(0, 1 / np.sqrt(D), (D, D)) for _ in range(3))
W_out = rng.normal(0, 1, (D, V))


def next_logits(ids, mask, pos):
    """Logits for the token after the last column of every row.

    ids, mask and pos are integer arrays of shape (B, T): the token ids, a mask that is 1
    where the token is real and 0 where it is a pad, and the position number of every column.
    """
    x = E[ids] + P[pos]                                   # (B, T, D)
    q = x[:, -1] @ Wq                                     # the last column asks
    k, v = x @ Wk, x @ Wv
    scores = np.einsum("bd,btd->bt", q, k) / np.sqrt(D)
    scores = np.where(mask == 1, scores, -np.inf)         # padding is never attended to
    w = np.exp(scores - scores.max(axis=1, keepdims=True))
    w /= w.sum(axis=1, keepdims=True)
    h = x[:, -1] + np.einsum("bt,btd->bd", w, v)
    logits = h @ W_out
    logits[:, PAD] = -np.inf                              # the model never writes a pad
    return logits


def generate_alone(prompt, n_new):
    """Greedy generation for one prompt with no padding: the reference."""
    ids = list(prompt)
    for _ in range(n_new):
        T = len(ids)
        logits = next_logits(np.array([ids]), np.ones((1, T), dtype=int), np.arange(T)[None, :])
        ids.append(int(np.argmax(logits[0])))
    return ids[len(prompt):]


prompts = [[3, 7, 1, 9, 4], [5, 2], [8, 8, 6], [11]]
for p in prompts:
    print(f"{str(p):<17} alone generates {generate_alone(p, 6)}")

## Part 1 (coding task): Put the prompts in one batch

Write `pad_batch(prompts, pad_id)` for a list of prompts of different lengths, each a list
of token ids. Return three integer arrays of shape `(B, T)`, where `B` is the number of
prompts and `T` the length of the longest one: the token ids with `pad_id` in the
unused places, the mask (1 for a real token, 0 for a pad) and the position number of
every column.

The model takes its prediction from the last column of each row, and its position
embeddings are learned, one vector per position number. The check passes when
`next_logits` gives every row of your batch the same logits as the prompt alone.

In [ ]:
def pad_batch(prompts, pad_id):
    """Token ids, mask and position numbers for a list of prompts, each of shape (B, T)."""
    # TODO
    raise NotImplementedError

In [ ]:
def alone_logits(p):
    T = len(p)
    return next_logits(np.array([p]), np.ones((1, T), dtype=int), np.arange(T)[None, :])[0]


def check_batch(batch_prompts):
    ids, mask, pos = (np.asarray(a) for a in pad_batch(batch_prompts, PAD))
    B, T = len(batch_prompts), max(len(p) for p in batch_prompts)
    assert ids.shape == mask.shape == pos.shape == (B, T), (
        f"expected three arrays of shape {(B, T)}, one row per prompt and one column per token "
        f"of the longest prompt, got {ids.shape}, {mask.shape}, {pos.shape}")
    logits = next_logits(ids, mask, pos)
    for i, p in enumerate(batch_prompts):
        real = mask[i] == 1
        assert real.sum() == len(p) and list(ids[i][real]) == list(p), (
            f"prompt {p}: the columns with mask 1 should hold its tokens in order, they hold {[int(t) for t in ids[i][real]]}")
        assert np.all(ids[i][~real] == PAD), f"prompt {p}: the padding columns should hold PAD"
        assert mask[i, -1] == 1, f"prompt {p}: its row ends on a pad, and the model reads the next token from the last column"
        assert [int(t) for t in pos[i][real]] == list(range(len(p))), (
            f"prompt {p}: its tokens get positions {[int(t) for t in pos[i][real]]} in the batch and {list(range(len(p)))} alone")
        assert np.allclose(logits[i], alone_logits(p)), f"prompt {p}: the batch gives different logits from the prompt alone"


check_batch(prompts)
check_batch([[2, 4], [9, 1, 1, 6, 3, 10, 5], [7]])
check_batch([[6, 6, 6]])
print("Part 1 passed")

## Part 2 (coding task): Generate for the whole batch

Write `generate_batch(prompts, n_new, pad_id)`. Start from `pad_batch`, then repeat
`n_new` times: call `next_logits` once on the whole batch, take the most likely token of
each row, and append those tokens to the batch as a new column. Return one list of the
`n_new` new tokens for each prompt. The check compares them with `generate_alone`.

In [ ]:
def generate_batch(prompts, n_new, pad_id):
    """Greedy generation for every prompt at once: one list of n_new new tokens per prompt."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_generation(batch_prompts, n_new):
    global next_logits
    calls, model = [], next_logits

    def counted(ids, mask, pos):
        calls.append(np.shape(ids))
        return model(ids, mask, pos)

    next_logits = counted
    try:
        got = generate_batch(batch_prompts, n_new, PAD)
    finally:
        next_logits = model
    assert len(calls) == n_new and all(s[0] == len(batch_prompts) for s in calls), (
        f"call next_logits once per new token on the whole batch: {len(calls)} calls with "
        f"batch sizes {[s[0] for s in calls]} for {n_new} new tokens and {len(batch_prompts)} prompts")
    for p, row in zip(batch_prompts, got):
        want = generate_alone(p, n_new)
        row = [int(t) for t in row]
        if row != want:
            k = next(j for j in range(n_new) if j >= len(row) or row[j] != want[j])
            hint = ("the first new token already differs" if k == 0 else
                    f"the first {k} new tokens agree and token {k + 1} does not: look at the mask and "
                    f"position the batch gives each token it appends")
            raise AssertionError(f"prompt {p}: the batch generates {row}, alone it generates {want} ({hint})")


check_generation(prompts, 6)
check_generation([[2, 4], [9, 1, 1, 6, 3, 10, 5], [7]], 8)
print("Part 2 passed")

## Solution

### Part 1: Put the prompts in one batch

In [ ]:
def pad_batch(prompts, pad_id):
    """Token ids, mask and position numbers for a list of prompts, each of shape (B, T)."""
    T = max(len(p) for p in prompts)
    ids = np.array([[pad_id] * (T - len(p)) + list(p) for p in prompts])
    mask = np.array([[0] * (T - len(p)) + [1] * len(p) for p in prompts])
    pos = np.maximum(np.cumsum(mask, axis=1) - 1, 0)      # 0 for the first real token of each row
    return ids, mask, pos

**Why it works.** `next_logits` takes its prediction from the last column, so every row has to end on
its own last token, and the padding goes on the left. The mask keeps the pads out of
attention. The position embeddings are learned, one vector per position number, so a
token's position number changes its vector: a prompt of length $n$ must sit at
positions $0, \dots, n - 1$ in the batch exactly as it does alone. The running count of
real tokens, `cumsum(mask) - 1`, gives that. Pads get position 0 here, and any number
would do, because nothing attends to them.

### Part 2: Generate for the whole batch

In [ ]:
def generate_batch(prompts, n_new, pad_id):
    """Greedy generation for every prompt at once: one list of n_new new tokens per prompt."""
    ids, mask, pos = pad_batch(prompts, pad_id)
    new = [[] for _ in prompts]
    for _ in range(n_new):
        nxt = np.argmax(next_logits(ids, mask, pos), axis=1)
        for row, token in zip(new, nxt):
            row.append(int(token))
        ids = np.column_stack([ids, nxt])
        mask = np.column_stack([mask, np.ones(len(prompts), dtype=int)])   # the new token is real
        pos = np.column_stack([pos, pos[:, -1] + 1])                      # one past each row's last
    return new

**Why it works.** Each appended token is a real token, so its mask entry is 1, or the next step could not
attend to it. Its position is one more than the last position of its own row, which is
the prompt's length plus the tokens generated so far. The column number would be wrong
for every row with padding, because the padding shifted that row's columns to the right
without shifting its positions. With both set this way, every row of the batch sees the
same tokens at the same positions as its prompt does alone, so it makes the same
choices, and the four prompts need 6 calls of the model instead of 24.